<a href="https://colab.research.google.com/github/BigFoots625/IntroductionMachineLearningwithPython/blob/main/Chapter5_ModelEvaluation_and_Improvment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Model Evaluation and Improvement**
Until now, we have evaluated our models using the `score` method, which typically returns accuracy for classification and $R^2$ for regression, based on a single train/test split.

However, a single split is fragile; it might luckily contain very "easy" examples in the test set, giving us false confidence. In this chapter, we will cover robust methods to evaluate generalization performance: **Cross-Validation**, **Grid Search** for parameter tuning, and **Evaluation Metrics** that go beyond simple accuracy.

In [ ]:
!pip install mglearn
import mglearn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

### **Theoretical Explanation: Cross-Validation**
**Cross-Validation (CV)** is a statistical method for evaluating generalization performance that is much more stable and thorough than using a single split.

In **k-fold cross-validation**, the data is divided into $k$ roughly equal parts (folds). A sequence of $k$ models is trained. The first model uses fold 1 as the test set and folds 2 through $k$ as the training set. The second model uses fold 2 as the test set, and so on. This ensures every single data point is in the test set exactly once.

There are several variations:
1.  **Stratified k-Fold:** Used for classification. It ensures that the proportion of classes in each fold is exactly the same as the proportion in the whole dataset.
2.  **Leave-One-Out (LOO):** Each fold is a single sample. Extremely computationally expensive but useful for very small datasets.
3.  **Shuffle-Split:** Samples `train_size` points for the training set and `test_size` points for the test set, and repeats this $n$ times. It allows for control over the number of iterations independent of the training and test sizes.
4.  **Group k-Fold:** Ensures that all data points belonging to the same defined "group" (like multiple medical records from the same patient) stay together in either the training or test set, preventing the model from artificially memorizing patient specifics.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, LeaveOneOut, ShuffleSplit
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris()
logreg = LogisticRegression(max_iter=1000)

# 1. Standard Cross-Validation (scikit-learn uses StratifiedKFold by default for classifiers)
scores = cross_val_score(logreg, iris.data, iris.target, cv=5)
print(f"Standard CV scores: {scores}")
print(f"Average CV score: {scores.mean():.3f}\n")

# 2. Standard KFold (Forces non-stratified splitting, which is bad for sorted datasets like Iris)
kfold = KFold(n_splits=3)
print(f"Standard KFold scores: {cross_val_score(logreg, iris.data, iris.target, cv=kfold)}")

# 3. Leave-One-Out CV
loo = LeaveOneOut()
scores_loo = cross_val_score(logreg, iris.data, iris.target, cv=loo)
print(f"\nLeave-One-Out CV average score: {scores_loo.mean():.3f} (over {len(scores_loo)} models)")

# 4. Shuffle-Split CV
shuffle_split = ShuffleSplit(test_size=.5, train_size=.5, n_splits=10)
scores_shuffle = cross_val_score(logreg, iris.data, iris.target, cv=shuffle_split)
print(f"\nShuffle-Split CV average score: {scores_shuffle.mean():.3f}")

### **Theoretical Explanation: Grid Search and Parameter Tuning**
Finding the right parameters (like $C$ for SVMs or `alpha` for Ridge) is an art. Trying them by hand is tedious. **Grid Search** exhaustively tries all possible combinations of a list of parameters you provide.

**The Danger of Overfitting Parameters:** If we use the test set to choose the best parameters, the test set "leaks" into the model. We can no longer use it to evaluate how well the model generalizes.
Therefore, we actually need *three* datasets:
1.  **Training set:** To build the model.
2.  **Validation set:** To select the parameters.
3.  **Test set:** To evaluate the final model's performance.

Because splitting data three ways leaves us with very little data to train on, we combine Grid Search with Cross-Validation (`GridSearchCV`). It performs cross-validation for *every single combination* of parameters.

In [ ]:
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

# Split the data into train+validation and test sets
X_trainval, X_test, y_trainval, y_test = train_test_split(iris.data, iris.target, random_state=0)

# Define the grid of parameters to search
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100],
              'gamma': [0.001, 0.01, 0.1, 1, 10, 100]}

# Instantiate GridSearchCV
grid_search = GridSearchCV(SVC(), param_grid, cv=5)

# Fit will run cross-validation on the trainval set for all 36 combinations of C and gamma
grid_search.fit(X_trainval, y_trainval)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.3f}")
print(f"Test set score with best parameters: {grid_search.score(X_test, y_test):.3f}")

### **Theoretical Explanation: Evaluation Metrics and Scoring**
Accuracy is often a terrible metric. If an dataset has 99 healthy patients and 1 cancer patient (an **imbalanced dataset**), a model that simply always guesses "Healthy" will be 99% accurate, but completely useless. We need better metrics:

**1. The Confusion Matrix:**
A $2 \times 2$ array showing the True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).

**2. Precision, Recall, and F1-Score:**
* **Precision:** Of all the points the model *predicted* as positive, how many were actually positive? Used when False Positives are very costly (e.g., predicting a drug trial is safe when it isn't).
    $$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
* **Recall (Sensitivity):** Of all the *actual* positive points, how many did the model find? Used when False Negatives are very costly (e.g., missing a cancer diagnosis).
    $$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}$$
* **F1-Score:** The harmonic mean of precision and recall. It is a much better metric than accuracy for imbalanced classes.
    $$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix, classification_report

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=0)

logreg = LogisticRegression(max_iter=10000).fit(X_train, y_train)
y_pred = logreg.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

### **Theoretical Explanation: Taking Uncertainty into Account (PR and ROC Curves)**
Classifiers don't just output absolute predictions; they output probabilities (`predict_proba`) or decision scores (`decision_function`). By default, a probability $> 0.5$ is classified as positive.

However, we can change this **operating point** (threshold). If we want to guarantee we catch every single case of cancer, we might lower the threshold to $0.1$.

To visualize how changing this threshold affects our model across *all* possible thresholds, we use two curves:
1.  **Precision-Recall Curve:** Plots Precision vs. Recall across all thresholds. We want a curve that stays as close to the top-right corner as possible.
2.  **ROC Curve (Receiver Operating Characteristic):** Plots the True Positive Rate (Recall) against the False Positive Rate (FPR). We want a curve that stays as close to the top-left corner as possible. We summarize this curve with a single number: **AUC (Area Under the Curve)**. An AUC of 1.0 is a perfect model; 0.5 is random guessing.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Calculate decision function scores
decision_function_scores = logreg.decision_function(X_test)

# Compute the ROC curve (FPR, TPR, and thresholds)
fpr, tpr, thresholds = roc_curve(y_test, decision_function_scores)

# Compute the AUC
auc_score = roc_auc_score(y_test, decision_function_scores)

# Plotting the ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.3f})")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR) / Recall")
plt.title("ROC Curve for Logistic Regression")
plt.legend(loc="best")
plt.show()

### **Theoretical Explanation: Metrics for Multiclass Classification**
When we have more than two classes (like the 3 classes in the Iris dataset), precision and recall get slightly more complicated. We still compute a confusion matrix, but it is now an $N \times N$ matrix.

To get a single number for precision, recall, or F1-score across all classes, we have to average them. There are three ways to do this:
1.  **Macro Average:** Computes the metric for each class independently, then takes the unweighted mean. It treats all classes equally, regardless of how many samples are in each class.
2.  **Weighted Average:** Computes the metric for each class, but weights the mean by the number of true instances in each class.
3.  **Micro Average:** Computes total True Positives, False Positives, and False Negatives globally across all classes, and computes the metric from those global totals. Used when we care equally about every single individual data point.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(digits.data, digits.target, random_state=0)

logreg_multi = LogisticRegression(max_iter=10000).fit(X_train, y_train)
y_pred = logreg_multi.predict(X_test)

print("Multiclass Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

print("\nMulticlass Classification Report:\n")
print(classification_report(y_test, y_pred))

### **Theoretical Explanation: Using Evaluation Metrics in Grid Search**
By default, `GridSearchCV` uses accuracy to find the best parameters. But if we are working on an imbalanced dataset (like fraud detection), optimizing for accuracy is dangerous. We want to optimize for the metric that actually solves our business problem.

Scikit-learn allows us to pass a `scoring` parameter to `GridSearchCV` (and `cross_val_score`) to explicitly tell it to use "roc_auc", "f1", "recall", or other custom metrics to find the best parameters.

In [ ]:
# Using the breast cancer dataset (imbalanced classes)
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state=0)

# Define the grid
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

# Grid search optimizing for standard Accuracy (default)
grid_acc = GridSearchCV(SVC(), param_grid, cv=3)
grid_acc.fit(X_train, y_train)

# Grid search optimizing explicitly for AUC
grid_auc = GridSearchCV(SVC(), param_grid, cv=3, scoring="roc_auc")
grid_auc.fit(X_train, y_train)

print(f"GridSearch finding parameters using Accuracy:")
print(f"Best parameters: {grid_acc.best_params_}")
print(f"Best cross-validation accuracy: {grid_acc.best_score_:.3f}")

print(f"\nGridSearch finding parameters using AUC:")
print(f"Best parameters: {grid_auc.best_params_}")
print(f"Best cross-validation AUC: {grid_auc.best_score_:.3f}")

### **Chapter 5 Summary**
* **Objective:** We moved beyond basic single-split accuracy to rigorously evaluate how models will perform in production on unseen data.
* **Key Concepts Covered:**
    * **Cross-Validation:** K-Fold, Stratified, Leave-One-Out, and Shuffle-Split methods for stable evaluation.
    * **Grid Search:** `GridSearchCV` for exhaustive hyperparameter tuning without leaking test data.
    * **Evaluation Metrics:** Confusion Matrices, Precision, Recall, and F1-scores to evaluate imbalanced datasets where accuracy fails.
    * **Uncertainty and Thresholds:** Utilizing `decision_function` and `predict_proba` to plot Precision-Recall and ROC curves (AUC) to adjust the model's operating points based on business needs.
* **Takeaway:** Building a model is only half the battle. If you cannot correctly measure its performance using the right metrics and validation strategies, you cannot trust its predictions.